# Common Built‑in Exceptions
- Python ships with a rich hierarchy of exception classes; most automation errors fall into a small, predictable subset.  
- All ordinary run‑time exceptions inherit from `Exception`, but subclasses convey *why* something failed (e.g., file missing vs. wrong type).  
- Catching overly broad bases like `Exception` hides root causes and can mask bugs—prefer the narrowest class you can handle.  
- Understanding the inheritance tree lets you decide when a single `except` can cover many related problems (e.g., `OSError`).  

In [10]:
import inspect, builtins

def show_err_tree(base, level = 0, max_depth = 1):
    if level > max_depth:
        return
    
    for name, obj in vars(builtins).items():
        if inspect.isclass(obj) and issubclass(obj, base) and obj is not base:
            print("\t" * level + f" - {name}")
            show_err_tree(obj, level + 1, max_depth)

show_err_tree(Exception, max_depth=4)

# import builtins
print(dir(builtins))

 - ArithmeticError
	 - FloatingPointError
	 - OverflowError
	 - ZeroDivisionError
 - AssertionError
 - AttributeError
 - BufferError
 - EOFError
 - ImportError
	 - ModuleNotFoundError
 - LookupError
	 - IndexError
	 - KeyError
 - MemoryError
 - NameError
	 - UnboundLocalError
 - OSError
	 - BlockingIOError
	 - ChildProcessError
	 - ConnectionError
		 - BrokenPipeError
		 - ConnectionAbortedError
		 - ConnectionRefusedError
		 - ConnectionResetError
	 - FileExistsError
	 - FileNotFoundError
	 - InterruptedError
	 - IsADirectoryError
	 - NotADirectoryError
	 - PermissionError
	 - ProcessLookupError
	 - TimeoutError
	 - BrokenPipeError
	 - ConnectionAbortedError
	 - ConnectionRefusedError
	 - ConnectionResetError
 - ReferenceError
 - RuntimeError
	 - NotImplementedError
	 - PythonFinalizationError
	 - RecursionError
 - StopAsyncIteration
 - StopIteration
 - SyntaxError
	 - IndentationError
		 - TabError
	 - _IncompleteInputError
	 - TabError
 - SystemError
 - TypeError
 - ValueError
	 - U

## `OSError` Family: Filesystem & Network Issues
- Signals problems interacting with the operating system: files, permissions, sockets, paths.  
- Subclasses such as `FileNotFoundError`, `PermissionError`, `IsADirectoryError`, `ConnectionRefusedError`, and `TimeoutError` offer granularity.  
- Catch individual subclasses when you can recover differently (create a missing file, prompt for sudo, retry a connection).  
- A single `except OSError` still groups all OS‑level failures when the response is the same (e.g., log and abort).  

In [12]:
try:
    with open('non_existing_file', 'r') as file:
        content = file.read()
except FileNotFoundError:
    print("File not found")
except PermissionError:
    print("Permission denied")
except OSError as os_err:
    print(f"Generel OS error: {os_err}")

File not found


## `KeyError`: Missing Dictionary Keys
- Raised when using `dict[key]` with a key that is absent.  
- Frequent in config loading, JSON parsing, or environment variable maps.  
- Mitigation patterns: `dict.get(key, default)`, membership tests (`if key in cfg`), or a tailored `except KeyError`.  
- Treats missing data distinctly from a wrong value (`ValueError`) or wrong type (`TypeError`).  

In [18]:
config = {"host": "server-01", "port": 8080}
config2 = {"host": "server-01", "port": 8080, "api_key": "1234545656"}
# api_key = config["api_key"]
api_key = config.get("api_key", "")
print(f"API key: {api_key}")


def call_endpoint(config, endpoint):
    """
    Calling the specified endpoint of the configured host.

    Args:
        config (dict(str)): Dict containing host, port and api_key
        endpoint (str): The endpoint to hit
    """
    if "api_key" in config:
        print(f"Making API call to endpoint: {endpoint} with key: {config["api_key"]}")
    else:
        print(f"No API key is available, not possible to call")

call_endpoint(config, "/users")
call_endpoint(config2, "/users")

def call_endpoint_exception(config, endpoint):
    """
    Calling the specified endpoint of the configured host.

    Args:
        config (dict(str)): Dict containing host, port and api_key
        endpoint (str): The endpoint to hit
    """
    try:
        print(f"Making API call to endpoint: {endpoint} with key: {config["api_key"]}")
    except KeyError as missing_key:
        print(f"No required key: {missing_key}, not possible to call")

call_endpoint_exception(config, "/users")
call_endpoint_exception(config2, "/users")


API key: 
No API key is available, not possible to call
Making API call to endpoint: /users with key: 1234545656
No required key: 'api_key', not possible to call
Making API call to endpoint: /users with key: 1234545656


## `IndexError`: Sequence Index Out of Bounds
- Triggered when list/tuple indices fall outside the valid range: negative beyond the left edge or ≥ `len(seq)`.  
- Common during iterative processing of dynamic lists or user‑provided indexes.  
- Prevent with bounds checks (`if i < len(seq)`), safe iteration (`for item in seq:`), or catch and default.  
- Signals "wrong position" rather than "wrong content".  

In [3]:
servers = ["web01", "web02"]

i =2

if i < len(servers):
    print(servers[i])

try:
    print(servers[i])
except IndexError as e:
    print(f"Index Error: {e}. List length is {len(servers)}")

Index Error: list index out of range. List length is 2


## `ValueError` vs. `TypeError`
- **ValueError**: argument type is acceptable but content/value is invalid (e.g., `int("abc")`).  
- **TypeError**: operation applied to an object of the wrong type altogether (e.g., `len(5)` or `"a" + 3`).  
- Distinguishing them clarifies whether to validate *content* or convert *types*.  
- Catch them separately to craft precise user feedback.  

In [6]:
try:
    port = int("http")
except ValueError as ve:
    print(f"Bad numeric string: {ve}")

try:
    total = "Errors" + 5
except TypeError as te:
    print(f"Type mismatch: {te}")

Bad numeric string: invalid literal for int() with base 10: 'http'
Type mismatch: can only concatenate str (not "int") to str


## `AttributeError`: Missing Object Member
- Raised when an attribute or method doesn't exist on the object referenced.  
- Often results from typos, unexpected `None`, or polymorphic functions returning different types.  
- Defensive techniques: `hasattr(obj, "attr")`, `if obj is not None:`, or narrow `except AttributeError`.  
- Conveys "object of this type doesn’t support that capability".  

## `ImportError` / `ModuleNotFoundError`
- Raised when an `import` statement cannot locate a module/package.  
- `ModuleNotFoundError` (Python 3.6+) is the specific subclass; catching `ImportError` also covers it.  
- Causes: misspelling, package not installed, wrong virtual environment, or PYTHONPATH issues.  
- Typical handling logs instructions and aborts early to avoid cascading failures.  

In [7]:
try:
    import nonExistentLib
except ModuleNotFoundError as me:
    print(f"Import failed: {me}")

Import failed: No module named 'nonExistentLib'
